# 05 — Inference & Enforcement Priority Ranking

**Prerequisite**: `notebooks/04_training.ipynb` must be complete.

**What happens here**:
1. Load the winning XGBoost checkpoint (auto-discovered from `configs/model.yaml`)
2. Rank all 140 zones for a specific date + hour → top-10 enforcement table
3. Generate a full-day enforcement schedule (morning + evening rush hours)
4. Save a self-contained HTML file with interactive Folium map + table

**Files saved**:
- `data/outputs/enforcement_priority_{date}_{hour}.html` — static demo output
- `data/outputs/day_schedule_{date}.csv` — hourly enforcement schedule

## Cell 1 — Environment setup
**Expected output**: `Project root: ...GridLock R2`

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\palur\OneDrive\Desktop\GridLock_R2_Transfer


## Cell 2 — Configure loguru
**Expected output**: `Loguru configured.`

In [2]:
import sys
from loguru import logger

logger.remove()
logger.add(
    sys.stdout,
    format='<green>{time:HH:mm:ss}</green> | <level>{level: <8}</level> | {message}',
    level='DEBUG',
    colorize=False,
)
print('Loguru configured.')

Loguru configured.


## Cell 3 — Load winning checkpoint
**What this cell does**: Auto-discovers the best checkpoint from `configs/model.yaml` (winner = xgboost/hour).  
**Expected output**: Checkpoint name, model name, resolution, 140 CIS zones loaded.

In [3]:
from src.inference.ranker import load_ranker

ranker = load_ranker(project_root=PROJECT_ROOT)

print(f"\nRanker loaded:")
print(f"  Model      : {ranker['model_name']}")
print(f"  Resolution : {ranker['time_resolution']}")
print(f"  Checkpoint : {ranker['ckpt_dir'].name}")
print(f"  CIS zones  : {len(ranker['cis_df'])}")
print(f"  Features   : {ranker['feature_cols']}")

20:57:36 | INFO     | Using checkpoint: 'xgboost_hour_20260618_151005'
20:57:36 | INFO     | Loading ranker: model=xgboost | resolution=hour | checkpoint=xgboost_hour_20260618_151005


Loading model weights:   0%|          | 0/5 [00:00<?, ?step/s]C:\Users\palur\OneDrive\Desktop\GridLock_R2_Transfer\venv\Lib\site-packages\xgboost\sklearn.py:1125: UserWarning: [20:57:37] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1509: Unknown file format: `xgb`. Using UBJSON (`ubj`) as a guess.
  self.get_booster().load_model(fname)
Loading zone stats:  60%|██████    | 3/5 [00:00<00:01,  1.31step/s]   

20:57:37 | INFO     |   zone_stats loaded from checkpoint: 140 zones


Building feature list: 100%|██████████| 5/5 [00:00<00:00,  6.20step/s]

20:57:37 | INFO     | ✓ Ranker loaded: xgboost/hour | 140 CIS zones | 18 features

Ranker loaded:
  Model      : xgboost
  Resolution : hour
  Checkpoint : xgboost_hour_20260618_151005
  CIS zones  : 140
  Features   : ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend', 'month', 'zone_mean_count', 'zone_median_count', 'zone_cis_score', 'zone_junction_frac', 'zone_total_count', 'fraction_at_junction', 'rolling_7d_count', 'dominant_violation_type', 'dominant_vehicle_type', 'violation_type_primary_encoded', 'vehicle_type_encoded', 'data_sent_to_scita_mean']


## Cell 4 — Rank zones for a specific date + hour
**What this cell does**: Scores all 140 zones for Monday 09:00 (morning rush) and returns the top-10 enforcement table.  

**Formula**: `priority_score(zone) = predicted_count(zone) × CIS(zone)`  

**Expected output**: Table with rank, zone_id, priority_score, tier (HIGH/MEDIUM/LOW).

In [4]:
from src.inference.ranker import rank_zones

TARGET_DATE = '2024-03-18'   # Monday (within test window)
TARGET_HOUR = 9              # Morning rush

top10 = rank_zones(
    ranker,
    target_date=TARGET_DATE,
    target_hour=TARGET_HOUR,
    top_k=10,
)

print(f'\n=== Top 10 Enforcement Zones — {TARGET_DATE} {TARGET_HOUR:02d}:00 ===')
print(top10[['zone_id', 'predicted_count', 'cis_score', 'priority_score', 'priority_tier', 'has_junction']].to_string())

20:57:37 | INFO     | Ranking zones: model=xgboost/hour | date=2024-03-18 hour=9 | top_k=10


Computing priority scores: 100%|██████████| 3/3 [00:00<00:00, 144.16step/s]

20:57:37 | INFO     | ✓ Ranking complete: top zone=2 | priority_score=76.1700 | tier=HIGH

=== Top 10 Enforcement Zones — 2024-03-18 09:00 ===
      zone_id  predicted_count  cis_score  priority_score priority_tier  has_junction
rank                                                                                 
1           2            50.78   1.500000         76.1700          HIGH             1
2           7             6.18   0.109697          0.6779           LOW             0
3          36            17.55   0.030277          0.5314           LOW             0
4           1             4.62   0.036384          0.1681           LOW             1
5          39             6.25   0.026418          0.1651           LOW             0
6          29             4.25   0.038595          0.1640           LOW             1
7           4             3.72   0.035253          0.1311           LOW             0
8           3             3.65   0.032513          0.1187           LOW            

## Cell 5 — Try different hours to see time-of-day variation
**What this cell does**: Rank zones at 3 different hours and show how the top-3 shifts.  
**Expected output**: Top-3 zones for early morning, rush hour, and evening.

In [5]:
import pandas as pd

hours_to_check = [7, 9, 12, 17, 20]
print(f'Time-of-day variation — {TARGET_DATE}')
print(f'{"Hour":<8} {"#1 Zone":<10} {"#1 Score":<12} {"#2 Zone":<10} {"#3 Zone":<10}')
print('-' * 55)

for h in hours_to_check:
    df_h = rank_zones(ranker, target_date=TARGET_DATE, target_hour=h, top_k=3)
    z = df_h['zone_id'].tolist()
    s = df_h['priority_score'].tolist()
    print(f"{h:02d}:00    {int(z[0]):<10} {s[0]:<12.4f} {int(z[1]):<10} {int(z[2])}")

Time-of-day variation — 2024-03-18
Hour     #1 Zone    #1 Score     #2 Zone    #3 Zone   
-------------------------------------------------------
20:57:37 | INFO     | Ranking zones: model=xgboost/hour | date=2024-03-18 hour=7 | top_k=3


Computing priority scores: 100%|██████████| 3/3 [00:00<00:00, 162.50step/s]

20:57:37 | INFO     | ✓ Ranking complete: top zone=2 | priority_score=84.2250 | tier=HIGH
07:00    2          84.2250      7          36
20:57:37 | INFO     | Ranking zones: model=xgboost/hour | date=2024-03-18 hour=9 | top_k=3



Computing priority scores: 100%|██████████| 3/3 [00:00<00:00, 169.50step/s]

20:57:37 | INFO     | ✓ Ranking complete: top zone=2 | priority_score=76.1700 | tier=HIGH
09:00    2          76.1700      7          36
20:57:37 | INFO     | Ranking zones: model=xgboost/hour | date=2024-03-18 hour=12 | top_k=3


...

20:57:37 | INFO     | ✓ Ranking complete: top zone=2 | priority_score=66.1650 | tier=HIGH
12:00    2          66.1650      7          36
20:57:37 | INFO     | Ranking zones: model=xgboost/hour | date=2024-03-18 hour=17 | top_k=3



Computing priority scores: 100%|██████████| 3/3 [00:00<00:00, 176.55step/s]

20:57:37 | INFO     | ✓ Ranking complete: top zone=2 | priority_score=69.4500 | tier=HIGH
17:00    2          69.4500      7          36
20:57:37 | INFO     | Ranking zones: model=xgboost/hour | date=2024-03-18 hour=20 | top_k=3



Computing priority scores: 100%|██████████| 3/3 [00:00<00:00, 176.00step/s]

20:57:37 | INFO     | ✓ Ranking complete: top zone=2 | priority_score=84.0900 | tier=HIGH
20:00    2          84.0900      7          36


## Cell 6 — Build zone centroids (needed for map)
**What this cell does**: Loads `features_with_zones.parquet` and computes mean lat/lon per zone.  
**Expected output**: 140 zone centroids, with Bengaluru coordinate range.

In [6]:
from src.inference.static_output import build_zone_centroids

centroids_df = build_zone_centroids(
    PROJECT_ROOT / 'data' / 'processed' / 'features_with_zones.parquet'
)

print(f'Zone centroids: {len(centroids_df)} zones')
print(f'Lat range : [{centroids_df["lat_centroid"].min():.4f}, {centroids_df["lat_centroid"].max():.4f}]')
print(f'Lon range : [{centroids_df["lon_centroid"].min():.4f}, {centroids_df["lon_centroid"].max():.4f}]')
print(centroids_df.head(5).to_string(index=False))

20:57:37 | INFO     | Zone centroids computed: 140 zones
Zone centroids: 140 zones
Lat range : [12.8193, 13.2448]
Lon range : [77.4670, 77.7629]
 zone_id  lat_centroid  lon_centroid
      -1     12.987365     77.605803
       0     12.924092     77.619629
       1     12.951224     77.529859
       2     12.978370     77.577252
       3     13.007835     77.693068


## Cell 7 — Generate static HTML output (demo fallback)
**What this cell does**: Generates a self-contained HTML file with Folium map + priority table.  
This is the **demo fallback** — if Streamlit crashes, open this file in any browser.  
**Expected output**: HTML file path and size.

In [7]:
from src.inference.static_output import generate_static_output

html_path = PROJECT_ROOT / 'data' / 'outputs' / f'enforcement_priority_{TARGET_DATE}_{TARGET_HOUR:02d}h.html'

out = generate_static_output(
    top_k_df        = top10,
    centroids_df    = centroids_df,
    target_date     = TARGET_DATE,
    target_hour     = TARGET_HOUR,
    output_path     = html_path,
    model_name      = ranker['model_name'],
    time_resolution = ranker['time_resolution'],
)

print(f'\nStatic HTML saved: {out}')
print(f'Size: {out.stat().st_size / 1e3:.1f} KB')
print('Open this file in your browser for the demo map.')

Building model scorecard: 100%|██████████| 4/4 [00:00<00:00, 18.74step/s]

20:57:38 | INFO     | ✓ Static output saved → 'C:\Users\palur\OneDrive\Desktop\GridLock_R2_Transfer\data\outputs\enforcement_priority_2024-03-18_09h.html' (45.1 KB)

Static HTML saved: C:\Users\palur\OneDrive\Desktop\GridLock_R2_Transfer\data\outputs\enforcement_priority_2024-03-18_09h.html
Size: 45.1 KB
Open this file in your browser for the demo map.


## Cell 8 — Full-day enforcement schedule
**What this cell does**: Generates top-5 zones for each rush-hour slot across the day.  
**Expected output**: Schedule DataFrame (hours × top-5 zones), saved as CSV.

In [8]:
from src.inference.ranker import rank_day_schedule

schedule_df = rank_day_schedule(
    ranker,
    target_date=TARGET_DATE,
    hours=[7, 8, 9, 10, 12, 14, 17, 18, 19, 20],
    top_k=5,
)

csv_path = PROJECT_ROOT / 'data' / 'outputs' / f'day_schedule_{TARGET_DATE}.csv'
schedule_df.to_csv(csv_path, index=False)

print(f'Day schedule saved: {csv_path}')
print(f'Shape: {schedule_df.shape}')
print()
print(schedule_df[['hour', 'rank', 'zone_id', 'predicted_count', 'priority_score', 'priority_tier']].to_string(index=False))

20:57:38 | INFO     | Generating day schedule: date=2024-03-18 | 10 hours | top_k=5


Ranking hours:   0%|          | 0/10 [00:00<?, ?hour/s]

20:57:38 | INFO     | Ranking zones: model=xgboost/hour | date=2024-03-18 hour=7 | top_k=5



Computing priority scores: 100%|██████████| 3/3 [00:00<00:00, 146.27step/s]

20:57:38 | INFO     | ✓ Ranking complete: top zone=2 | priority_score=84.2250 | tier=HIGH
20:57:38 | INFO     | Ranking zones: model=xgboost/hour | date=2024-03-18 hour=8 | top_k=5


...

20:57:38 | INFO     | Ranking zones: model=xgboost/hour | date=2024-03-18 hour=20 | top_k=5



Computing priority scores: 100%|██████████| 3/3 [00:00<00:00, 165.94step/s]

20:57:38 | INFO     | ✓ Ranking complete: top zone=2 | priority_score=84.0900 | tier=HIGH



Ranking hours: 100%|██████████| 10/10 [00:00<00:00, 40.98hour/s]

20:57:38 | INFO     | ✓ Day schedule: 50 rows (10 hours × 5 zones)
Day schedule saved: C:\Users\palur\OneDrive\Desktop\GridLock_R2_Transfer\data\outputs\day_schedule_2024-03-18.csv
Shape: (50, 9)

 hour  rank  zone_id  predicted_count  priority_score priority_tier
    7     1        2            56.15         84.2250          HIGH
    7     2        7             6.45          0.7075           LOW
    7     3       36            18.89          0.5719           LOW
    7     4        1             5.06          0.1841           LOW
    7     5       39             6.57          0.1736           LOW
    8     1        2            52.66         78.9900          HIGH
    8     2        7             6.07          0.6659           LOW
    8     3       36            17.78          0.5383           LOW
    8     4        1             4.81          0.1750           LOW
    8     5       29             4.35          0.1679           LOW
    9     1        2            50.78         76.1700  

## Summary

**What was done**:
- Winning XGBoost/hour checkpoint loaded from `checkpoints/`
- All 140 zones scored for a specific date/hour using `priority_score = predicted_count × CIS`
- Time-of-day variation explored (zone rankings shift by hour)
- Self-contained HTML output generated (demo fallback)
- Full-day enforcement schedule saved as CSV

**Files saved**:
- `data/outputs/enforcement_priority_{date}_{hour}h.html`
- `data/outputs/day_schedule_{date}.csv`

**Next step**: `src/data/pipeline.py` — end-to-end orchestrator (runs steps 1→8 in one command)

In [9]:
print('=== Inference Summary ===')
print(f'  Model          : {ranker["model_name"]}')
print(f'  Resolution     : {ranker["time_resolution"]}')
print(f'  Target date    : {TARGET_DATE}')
print(f'  Target hour    : {TARGET_HOUR:02d}:00')
print(f'  Zones scored   : {len(ranker["cis_df"])}')
print(f'  Top zone       : {int(top10.iloc[0]["zone_id"])} (score={top10.iloc[0]["priority_score"]:.4f}, tier={top10.iloc[0]["priority_tier"]})')
print(f'  HTML output    : data/outputs/enforcement_priority_{TARGET_DATE}_{TARGET_HOUR:02d}h.html')
print(f'  Schedule CSV   : data/outputs/day_schedule_{TARGET_DATE}.csv')
print(f'  Next step      : src/data/pipeline.py (end-to-end orchestrator)')

=== Inference Summary ===
  Model          : xgboost
  Resolution     : hour
  Target date    : 2024-03-18
  Target hour    : 09:00
  Zones scored   : 140
  Top zone       : 2 (score=76.1700, tier=HIGH)
  HTML output    : data/outputs/enforcement_priority_2024-03-18_09h.html
  Schedule CSV   : data/outputs/day_schedule_2024-03-18.csv
  Next step      : src/data/pipeline.py (end-to-end orchestrator)
